In [ ]:
import os, sys
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
# Cria a conexão Spark

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [ ]:
PROJECT_PATH    = os.getcwd()
ROOT_DATA_PATH  = "C:\\Marco Conti\\Projetos\\Dados\\"

In [ ]:
def write_data(df_, write_path, prefix_file_name):
    # df_temperatura_final.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_t2m_temperatura.csv", index=False)
    df_.toPandas().to_parquet(f"{write_path}\\{prefix_file_name}.parquet")

# Somente para processamento LOCAL, pode ser descartada
def write_data_csv(df_write, write_path, file_name):
    df_write.toPandas().to_csv(f"{write_path}\{file_name}", index=False)

Seleciona os dados de temperatura

In [ ]:
from pathlib import Path
import re

pasta_t2m_parquet = Path(r"C:\Marco Conti\Projetos\Dados\ERA5-temperaturas\arquivos_parquet")

rec = 0
for arquivo_parquet in pasta_t2m_parquet.glob("*.parquet"):

    path = r"{0}".format(arquivo_parquet)
    ano = int(re.search(r'(\d{4})(?=\.parquet$)', path).group(1))

    if ano >= 1995:
        
        df_ = spark.read.parquet(path)
        if rec == 0:
            df_tempetatura = df_
        else:
            df_tempetatura = df_tempetatura.union(df_)

        rec += 1

Obtem as estatísticas de temperatura por Mês:
- Temperatura mínima
- Temperatura máxima
- Temperatura média

In [ ]:
# Criar as colunas min, max, med por MES*
# Estas estatísticas já são os primeiros 3 indicadores

df_base_mes = (
    df_tempetatura
        .withColumn("ano", F.year("data_medicao"))
        .withColumn("mes", F.month("data_medicao"))
)

df_stats_mes = (
    df_base_mes
    .groupBy("ano"
           ,"mes"
            ,"latitude"
            ,"longitude"
    )
    .agg(F.min("valor").alias("temp_min_mes")
        ,F.max("valor").alias("temp_max_mes")
        ,F.avg("valor").alias("temp_media_mes")
    )
)

# print("df_stats.count()", df_stats.count())
df_stats_mes.printSchema()
# df_stats.show(10, truncate=False)

In [ ]:
df_stats_mes_normalize_datatype = \
    (df_stats_mes
        .withColumns({"temp_min_mes":F.col("temp_min_mes").cast("double")
                     ,"temp_max_mes": F.col("temp_max_mes").cast("double")
                     ,"temp_media_mes": F.col("temp_media_mes").cast("double")
                     }))

# Transformar a colunas de estatísticas em linhas
df_stats_mes_transpose = (
    df_stats_mes_normalize_datatype.select(
        "ano",
        "mes",
        "latitude",
        "longitude",
        F.expr("""
            stack(
                3,
                'Temperatura mínima', temp_min_mes,
                'Temperatura máxima', temp_max_mes,
                'Temperatura média',  temp_media_mes
            ) as (indicador, valor)
        """),
        F.lit("Celsius").alias("unidade_medida")
    )
)

df_stats_mes_transpose.printSchema()

# Grava somente os últimos 10 anos
df_stats_mes_transpose_write = df_stats_mes_transpose.filter("ano >= 2015")

write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
prefix_file_name = "temperatura_min_med_max_ano_mes"

write_data(df_stats_mes_transpose_write, write_path, prefix_file_name)

Obtem as estatísticas de temperatura por Ano:
- Temperatura mínima
- Temperatura máxima
- Temperatura média
- Percentil 5%
- Percentil 90%

In [ ]:
# Criar as colunas min, max, med e percentis por ANO*, que serão utilizadas para apurar os
# valores extremos (calor e frio) e os 6 indicadores de Ondas de Calor e Ondas de Frio
# * De acordo com documento elaborado por Sara Lopes de Moraes para o VERACIS

df_base = (
    df_tempetatura
        .withColumn("ano", F.year("data_medicao"))
        .withColumn("mes", F.month("data_medicao"))
)

# df_base.printSchema()
# df_base.show(10, truncate=False)

df_stats = (
    df_base
    .groupBy("ano"
            ,"latitude"
            ,"longitude"
    )
    .agg(F.min("valor").alias("temp_min_ano")
        ,F.max("valor").alias("temp_max_ano")
        ,F.avg("valor").alias("temp_media_ano")
        ,F.expr("percentile_approx(valor, 0.05)").alias("percentil_05_ano")
        ,F.expr("percentile_approx(valor, 0.95)").alias("percentil_95_ano")
    )
)

# print("df_stats.count()", df_stats.count())
df_stats.printSchema()
# df_stats.show(10, truncate=False)

In [ ]:
df_stats_ano_normalize_datatype = \
    (df_stats
        .withColumns({"temp_min_ano":F.col("temp_min_ano").cast("double")
                     ,"temp_max_ano": F.col("temp_max_ano").cast("double")
                     ,"temp_media_ano": F.col("temp_media_ano").cast("double")
                     }))


df_stats_ano_transpose = (
    df_stats_ano_normalize_datatype.select(
        "ano",
        F.lit(0).alias("mes"),
        "latitude",
        "longitude",
        F.expr("""
            stack(
                3,
                'Temperatura mínima', temp_min_ano,
                'Temperatura máxima', temp_max_ano,
                'Temperatura média',  temp_media_ano
            ) as (indicador, valor)
        """),
        F.lit("Celsius").alias("unidade_medida")
    )
)

df_stats_ano_transpose.printSchema()

# Grava somente os últimos 10 anos
df_stats_ano_transpose_write = df_stats_ano_transpose.filter("ano >= 2015")

write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
prefix_file_name = "temperatura_min_med_max_ano"

write_data(df_stats_ano_transpose_write, write_path, prefix_file_name)

Anexar dados estatíscos e percentis ao dado diário

Será utilizado para calcular as ondas de calor (periodos de dias consecutivos)

In [ ]:
df_dia = (
    df_base.alias("base")
    .join(
        df_stats.alias("stats"),
        [
            F.col("base.ano") == F.col("stats.ano"),
            F.col("base.latitude") == F.col("stats.latitude"),
            F.col("base.longitude") == F.col("stats.longitude"),
        ],
        how="left",
    )
    .select(
        "stats.ano",
        "base.mes",
        "stats.latitude",
        "stats.longitude",
        "stats.temp_min_ano",
        "stats.temp_max_ano",
        "stats.temp_media_ano",
        "stats.percentil_05_ano",
        "stats.percentil_95_ano",
        "base.data_medicao",
        "base.indicador",
        "base.valor",
        "base.unidade_medida"
    )
)

# print("Número de registros no DataFrame diário:", df_dia.count()) # 151662
# df_dia.printSchema()
# df_dia.filter("ano = 2023 and latitude = -23 and longitude = -46").orderBy("valor").show(10, truncate=False)

In [ ]:
# Número de registros no DataFrame diário: 27.703.592

Classifica os valores extremos para calor e frio
- Calor: temperatura diaria é maior ou igual ao percentil 95
- Frio: temperatura diária é menor ou igual ao percentil 5

In [ ]:
df_dia = (
    df_dia
    .withColumn(
        "extremo_alto",
        F.when(F.col("valor") >= F.col("percentil_95_ano")
              ,(F.col("valor") - F.col("percentil_95_ano"))).otherwise(0))
    .withColumn(
        "extremo_baixo",
        F.when(F.col("valor") <= F.col("percentil_05_ano")
              ,(F.col("percentil_05_ano") - F.col("valor"))).otherwise(0)
    )
)

Separa somente os dias quentes ou frios de acordo com a regra definida no passo anterior

In [ ]:

df_flag = \
    (df_dia.withColumn("flag_dia_quente"
                     ,F.when(F.col("valor") > F.col("percentil_95_ano"), 1).otherwise(0))
           .withColumn("flag_dia_frio"
                      ,F.when(F.col("valor") < F.col("percentil_05_ano"), 1).otherwise(0)))

# Mantém apenas os dias que atenderam ao critério de dias quentes ou frios
df_quentes = df_flag.filter(F.col("flag_dia_quente") == 1)
df_frios   = df_flag.filter(F.col("flag_dia_frio") == 1)

# Janela ordenada por data para cada ponto geográfico, assim será possível identificar os períodos consecutivos de ondas de calor ou frio
janela_loc = Window.partitionBy("latitude", "longitude").orderBy("data_medicao")

# Ao subtrair a ordem do registro (rn) da data, dias consecutivos geram
# exatamente o mesmo identificador de grupo (grupo_id)
df_eventos = \
    (df_quentes
        .withColumn("rn", F.row_number().over(janela_loc)) \
        .withColumn("grupo_id", F.expr("date_sub(data_medicao, CAST(rn AS INT))")))

df_eventos_frio = \
    (df_frios
        .withColumn("rn", F.row_number().over(janela_loc))
        .withColumn("grupo_id", F.expr("date_sub(data_medicao, CAST(rn AS INT))"))
)


In [ ]:
# Identificação dos Eventos de CALOR e Validação da Duração (Global, sem quebra de mês/ano, usando apenas as coordenadas geográfica)
df_eventos_duracao = (
    df_eventos
    .groupBy("latitude", "longitude", "grupo_id")
    .agg(
        F.count("data_medicao").alias("duracao_total_onda"),
        F.min("data_medicao").alias("inicio_onda"),
        F.max("data_medicao").alias("fim_onda")
    )
    .filter(F.col("duracao_total_onda") > 2) # Filtra apenas eventos reais (> 2 dias)
)

# 2. Retornar os DIAS INDIVIDUAIS das ondas de calor válidas mantendo os metadados da onda
df_dias_em_onda = (
    df_eventos
    .join(df_eventos_duracao, ["latitude", "longitude", "grupo_id"], "inner")
    .select("latitude", 
            "longitude", 
            "data_medicao", 
            "ano", 
            "mes", 
            "grupo_id", 
            "duracao_total_onda",
            "valor"
    )
)

# df_dias_em_onda.filter("latitude = -23 and longitude = -46").show(10, truncate=False)

In [ ]:
# Identificação dos Eventos de FRIO e Validação da Duração (Global, sem quebra de mês/ano, usando apenas as coordenadas geográfica)
df_eventos_duracao_frio = (
    df_eventos_frio
    .groupBy("latitude", "longitude", "grupo_id")
    .agg(
        F.count("data_medicao").alias("duracao_total_onda"),
        F.min("data_medicao").alias("inicio_onda"),
        F.max("data_medicao").alias("fim_onda")
    )
    .filter(F.col("duracao_total_onda") > 2) # Filtra apenas eventos reais (> 2 dias)
)

# 2. Dias individuais de ondas de frio válidas
df_dias_em_onda_frio = (
    df_eventos_frio
    .join(df_eventos_duracao_frio, ["latitude", "longitude", "grupo_id"], "inner")
    .select(
        "latitude", 
        "longitude", 
        "data_medicao", 
        "ano", 
        "mes", 
        "grupo_id", 
        "duracao_total_onda",
        "valor"  # Mão dupla: temperatura para amplitude e magnitude
    )
)

#### Adicionar as métricas:

<pre>
- Número de ondas de calor (N-OdC)      : Total de eventos de ondas de calor registrados em um determinado ano.
- Frequência das ondas de calor (F-OdC) : Número total de dias que compõem as ondas de calor ao longo do ano.
- Duração das ondas de calor (D-OdC)    : Duração, em dias, do evento de onda de calor mais longo registrado no ano.
- Amplitude das ondas de calor (A-OdC)  : Maior valor da temperatura média diária observado durante eventos de onda de calor no ano.
- Magnitude das ondas de calor (M-OdC)  : Média da temperatura média diária considerando todos os dias de ocorrência de ondas de calor no ano.
</pre>

In [ ]:
# Métricas mensais para Ondas de CALOR

df_metricas_mensal = (
    df_dias_em_onda
    .groupBy("latitude", "longitude", "ano", "mes")
    .agg(
        # N-OdC: Número de ondas distintas no mês
        F.countDistinct("grupo_id").alias("numero_ondas_calor"),
        
        # F-OdC: Total exato de dias sob onda de calor DENTRO do mês
        F.count("data_medicao").alias("frequencia_dias_onda_calor"),
        
        # D-OdC: Duração máxima (e média) das ondas no mês
        F.max("duracao_total_onda").alias("duracao_maxima_onda"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas"),
        
        # A-OdC: Maior temperatura diária observada em dias de onda de calor no mês
        F.max("valor").alias("amplitude_onda_calor"),
        
        # M-OdC: Média das temperaturas diárias considerando os dias de onda de calor no mês
        F.round(F.avg("valor"), 2).alias("magnitude_onda_calor")
    )
    .orderBy("ano", "mes", "latitude", "longitude")
)


df_metricas_mensal.printSchema()

# df_metricas_mensal.limit(10).show(truncate=False)

# write_data(df_metricas_mensal, ROOT_DATA_PATH, "Ondas_Calor_Mensal")




In [ ]:
df_metricas_mensal_normalize_datatype = \
    (df_metricas_mensal
        .withColumns({"numero_ondas_calor":F.col("numero_ondas_calor").cast("double")
                     ,"frequencia_dias_onda_calor": F.col("frequencia_dias_onda_calor").cast("double")
                     ,"duracao_maxima_onda": F.col("duracao_maxima_onda").cast("double")
                     ,"amplitude_onda_calor": F.col("amplitude_onda_calor").cast("double")
                     ,"amplitude_onda_calor": F.col("amplitude_onda_calor").cast("double")
                     }))


df_metricas_mensal_transpose = (
    df_metricas_mensal_normalize_datatype.select(
        "ano",
        "mes",
        "latitude",
        "longitude",
        F.expr("""
            stack(
                5,
                'Número de ondas de calor', numero_ondas_calor, 'eventos',
                'Frequência das ondas de calor', frequencia_dias_onda_calor, 'dias',
                'Duração das ondas de calor', duracao_maxima_onda, 'dias',
                'Amplitude das ondas de calor', amplitude_onda_calor, 'Celsius',
                'Magnitude das ondas de calor', magnitude_onda_calor, 'Celsius'
            ) as (indicador, valor, unidade_medida)
        """)
    )
)

df_metricas_mensal_transpose.printSchema()

# df_metricas_mensal_transpose.show(truncate=False)

# Grava somente os últimos 10 anos
df_metricas_mensal_transpose_write = df_metricas_mensal_transpose.filter("ano >= 2015")

write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
prefix_file_name = "temperatura_ondas_calor_ano_mes"

write_data(df_metricas_mensal_transpose_write, write_path, prefix_file_name)


In [ ]:
# Métricas mensais para Ondas de FRIO

df_metricas_mensal_frio = (
    df_dias_em_onda_frio
    .groupBy("latitude", "longitude", "ano", "mes")
    .agg(
        # N-OdF: Número de ondas de frio distintas
        F.countDistinct("grupo_id").alias("numero_ondas_frio"),
        
        # F-OdF: Frequência total de dias em onda de frio no mês
        F.count("data_medicao").alias("frequencia_dias_onda_frio"),
        
        # D-OdF: Duração máxima (e média) das ondas no mês
        F.max("duracao_total_onda").alias("duracao_maxima_onda_frio"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas_frio"),
        
        # A-OdF: Menor temperatura diária observada durante as ondas no mês
        F.min("valor").alias("amplitude_onda_frio"),
        
        # M-OdF: Média das temperaturas diárias durante os dias de onda de frio
        F.round(F.avg("valor"), 2).alias("magnitude_onda_frio")
    )
    .orderBy("ano", "mes", "latitude", "longitude")
)

df_metricas_mensal_frio.printSchema()

# df_metricas_mensal_frio.limit(10).show(truncate=False)

# write_data(df_metricas_mensal_frio, ROOT_DATA_PATH, "Ondas_Frio_Mensal")

# df_metricas_mensal_frio.filter("latitude = -23 and longitude = -46").show(10, truncate=False)


In [45]:
df_metricas_mensal_frio_normalize_datatype = \
    (df_metricas_mensal_frio
        .withColumns({"numero_ondas_frio":F.col("numero_ondas_frio").cast("double")
                     ,"frequencia_dias_onda_frio": F.col("frequencia_dias_onda_frio").cast("double")
                     ,"duracao_maxima_onda_frio": F.col("duracao_maxima_onda_frio").cast("double")
                     ,"amplitude_onda_frio": F.col("amplitude_onda_frio").cast("double")
                     ,"amplitude_onda_frio": F.col("amplitude_onda_frio").cast("double")
                     }))

df_metricas_mensal_frio_transpose = (
    df_metricas_mensal_frio_normalize_datatype.select(
        "ano",
        "mes",
        "latitude",
        "longitude",
        F.expr("""
            stack(
                5,
                'Número de ondas de frio', numero_ondas_frio, 'eventos',
                'Frequência das ondas de frio', frequencia_dias_onda_frio, 'dias',
                'Duração das ondas de frio', duracao_maxima_onda_frio, 'dias', 
                'Amplitude das ondas de frio', amplitude_onda_frio, 'Celsius',
                'Magnitude das ondas de frio', magnitude_onda_frio, 'Celsius'
            ) as (indicador, valor, unidade_medida)
        """)
    )
)

df_metricas_mensal_frio_transpose.printSchema()

# Grava somente os últimos 10 anos
df_metricas_mensal_frio_transpose_write = df_metricas_mensal_frio_transpose.filter("ano >= 2015")

write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
prefix_file_name = "temperatura_ondas_frio_ano_mes"

write_data(df_metricas_mensal_frio_transpose_write, write_path, prefix_file_name)


In [46]:
# Métricas anuais para Ondas de CALOR

df_metricas_anual = (
    df_dias_em_onda
    .groupBy("latitude", "longitude", "ano")
    .agg(
        # N-OdC: Número de ondas distintas no ano
        F.countDistinct("grupo_id").alias("numero_ondas_calor"),
        
        # F-OdC: Total de dias do ano passados sob onda de calor
        F.count("data_medicao").alias("frequencia_dias_onda_calor"),
        
        # D-OdC: Duração máxima do evento no ano
        F.max("duracao_total_onda").alias("duracao_maxima_onda"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas"),
        
        # A-OdC: Maior temperatura média diária observada durante ondas no ano
        F.max("valor").alias("amplitude_onda_calor"),
        
        # M-OdC: Média das temperaturas diárias em todos os dias de onda de calor no ano
        F.round(F.avg("valor"), 2).alias("magnitude_onda_calor")
    )
    .orderBy("ano", "latitude", "longitude")
)

df_metricas_anual.printSchema()

# df_metricas_anual.show(10,False)

# write_data(df_metricas_anual, ROOT_DATA_PATH, "Ondas_Calor_Anual")

# df_metricas_anual.filter("latitude = -23 and longitude = -46").show(10, truncate=False)

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- ano: integer (nullable = true)
 |-- numero_ondas_calor: long (nullable = false)
 |-- frequencia_dias_onda_calor: long (nullable = false)
 |-- duracao_maxima_onda: long (nullable = true)
 |-- duracao_media_ondas: double (nullable = true)
 |-- amplitude_onda_calor: double (nullable = true)
 |-- magnitude_onda_calor: double (nullable = true)



In [47]:
df_metricas_anual_normalize_datatype = \
    (df_metricas_anual
        .withColumns({"numero_ondas_calor":F.col("numero_ondas_calor").cast("double")
                     ,"frequencia_dias_onda_calor": F.col("frequencia_dias_onda_calor").cast("double")
                     ,"duracao_maxima_onda": F.col("duracao_maxima_onda").cast("double")
                     ,"amplitude_onda_calor": F.col("amplitude_onda_calor").cast("double")
                     ,"amplitude_onda_calor": F.col("amplitude_onda_calor").cast("double")
                     }))

df_metricas_anual_transpose = (
    df_metricas_anual_normalize_datatype.select(
        "ano",
        F.lit(0).alias("mes"),
        "latitude",
        "longitude",
        F.expr("""
            stack(
                5,
                'Número de ondas de calor', numero_ondas_calor, 'eventos', 
                'Frequência das ondas de calor', frequencia_dias_onda_calor, 'dias', 
                'Duração das ondas de calor', duracao_maxima_onda, 'dias',
                'Amplitude das ondas de calor', amplitude_onda_calor, 'Celsius', 
                'Magnitude das ondas de calor', magnitude_onda_calor, 'Celsius'
            ) as (indicador, valor, unidade_medida)

        """)
    )
)

df_metricas_anual_transpose.printSchema()

# Grava somente os últimos 10 anos
df_metricas_anual_transpose_write = df_metricas_anual_transpose.filter("ano >= 2015")

write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
prefix_file_name = "temperatura_ondas_calor_ano"

write_data(df_metricas_anual_transpose_write, write_path, prefix_file_name)


root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = false)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)



In [48]:
# Métricas anuais para Ondas de FRIO

df_metricas_anual_frio = (
    df_dias_em_onda_frio
    .groupBy("latitude", "longitude", "ano")
    .agg(
        # N-OdF: Número de ondas de frio distintas no ano
        F.countDistinct("grupo_id").alias("numero_ondas_frio"),
        
        # F-OdF: Total de dias do ano passados sob onda de frio
        F.count("data_medicao").alias("frequencia_dias_onda_frio"),
        
        # D-OdF: Duração máxima (e média) do evento no ano
        F.max("duracao_total_onda").alias("duracao_maxima_onda_frio"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas_frio"),
        
        # A-OdF: Menor temperatura diária registrada durante ondas de frio no ano
        F.min("valor").alias("amplitude_onda_frio"),
        
        # M-OdF: Média das temperaturas nos dias sob onda de frio no ano
        F.round(F.avg("valor"), 2).alias("magnitude_onda_frio")
    )
    .orderBy("ano", "latitude", "longitude")
)

df_metricas_anual_frio.printSchema()

# df_metricas_anual_frio.show(10, False)

# write_data(df_metricas_anual_frio, ROOT_DATA_PATH, "Ondas_Frio_Anual")

# df_metricas_anual_frio.filter("latitude = -23 and longitude = -46").show(10, truncate=False)

root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- ano: integer (nullable = true)
 |-- numero_ondas_frio: long (nullable = false)
 |-- frequencia_dias_onda_frio: long (nullable = false)
 |-- duracao_maxima_onda_frio: long (nullable = true)
 |-- duracao_media_ondas_frio: double (nullable = true)
 |-- amplitude_onda_frio: double (nullable = true)
 |-- magnitude_onda_frio: double (nullable = true)



In [49]:
df_metricas_anual_frio_normalize_datatype = \
    (df_metricas_anual_frio
        .withColumns({"numero_ondas_frio":F.col("numero_ondas_frio").cast("double")
                     ,"frequencia_dias_onda_frio": F.col("frequencia_dias_onda_frio").cast("double")
                     ,"duracao_maxima_onda_frio": F.col("duracao_maxima_onda_frio").cast("double")
                     ,"amplitude_onda_frio": F.col("amplitude_onda_frio").cast("double")
                     ,"amplitude_onda_frio": F.col("amplitude_onda_frio").cast("double")
                     }))

df_metricas_ano_frio_transpose = (
    df_metricas_anual_frio_normalize_datatype.select(
        "ano",
        F.lit(0).alias("mes"),
        "latitude",
        "longitude",
        F.expr("""
            stack(
                5,
                'Número de ondas de frio', numero_ondas_frio, 'eventos',
                'Frequência das ondas de frio', frequencia_dias_onda_frio, 'dias',
                'Duração das ondas de frio', duracao_maxima_onda_frio, 'dias',
                'Amplitude das ondas de frio', amplitude_onda_frio, 'Celsius',
                'Magnitude das ondas de frio', magnitude_onda_frio, 'Celsius'
            ) as (indicador, valor, unidade_medida)
        """)
    )
)

df_metricas_ano_frio_transpose.printSchema()

# Grava somente os últimos 10 anos
df_metricas_ano_frio_transpose_write = df_metricas_ano_frio_transpose.filter("ano >= 2015")

write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
prefix_file_name = "temperatura_ondas_frio_ano"

write_data(df_metricas_ano_frio_transpose_write, write_path, prefix_file_name)


root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = false)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)



In [57]:
df_temp_estat_ano_mes   = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_min_med_max_ano_mes.parquet")
df_temp_estat_ano_mes.createOrReplaceTempView("temp_temp_estat_ano_mes")
df_temp_estat_ano       = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_min_med_max_ano.parquet")
df_temp_estat_ano.createOrReplaceTempView("temp_temp_estat_ano")
df_ondas_calor_ano_mes  = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_ondas_calor_ano_mes.parquet")
df_ondas_calor_ano_mes.createOrReplaceTempView("temp_ondas_calor_ano_mes")
df_ondas_calor_ano      = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_ondas_calor_ano.parquet")
df_ondas_calor_ano.createOrReplaceTempView("temp_ondas_calor_ano")
df_ondas_frio_ano_mes   = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_ondas_frio_ano_mes.parquet")
df_ondas_frio_ano_mes.createOrReplaceTempView("temp_ondas_frio_ano_mes")
df_ondas_frio_ano_mes   = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_ondas_frio_ano.parquet")
df_ondas_frio_ano_mes.createOrReplaceTempView("temp_ondas_frio_ano")

In [71]:
query = \
    """with ind_clima as (
            Select * from temp_temp_estat_ano_mes
            Union All
            Select * from temp_temp_estat_ano
            Union All 
            Select * from temp_ondas_calor_ano_mes
            Union All
            Select * from temp_ondas_calor_ano
            Union All
            Select * from temp_ondas_frio_ano_mes
            Union All
            Select * from temp_ondas_frio_ano)
        Select * 
          from ind_clima
         where 1=1
           and ano = 2025
           and mes = 6
           and latitude > -24
           and latitude < -23
           and longitude > -47
           and longitude < -46
           and indicador rlike 'Temperatura'
         order by mes, indicador
    """

spark.sql(query).show(100,False)

+----+---+--------+---------+------------------+------------------+--------------+
|ano |mes|latitude|longitude|indicador         |valor             |unidade_medida|
+----+---+--------+---------+------------------+------------------+--------------+
|2025|6  |-23.25  |-46.75   |Temperatura máxima|17.711877441406273|Celsius       |
|2025|6  |-23.5   |-46.25   |Temperatura máxima|17.985864257812523|Celsius       |
|2025|6  |-23.75  |-46.75   |Temperatura máxima|18.822595214843773|Celsius       |
|2025|6  |-23.75  |-46.5    |Temperatura máxima|18.693261718750023|Celsius       |
|2025|6  |-23.5   |-46.5    |Temperatura máxima|18.397424316406273|Celsius       |
|2025|6  |-23.5   |-46.75   |Temperatura máxima|18.577111816406273|Celsius       |
|2025|6  |-23.75  |-46.25   |Temperatura máxima|18.883264160156273|Celsius       |
|2025|6  |-23.25  |-46.5    |Temperatura máxima|18.090783691406273|Celsius       |
|2025|6  |-23.25  |-46.25   |Temperatura máxima|18.704614257812523|Celsius       |
|202

In [66]:
4.96 / 0.8

6.199999999999999